In [ ]:
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import torch

# Add src to path
sys.path.append(os.path.abspath(".."))

from src.loss_2d import BiGaussianLoss
from src.optimize_2d import compute_normals, sample_profiles, smooth_contour

In [ ]:
# 1. Load Data
data_path = Path("../data/2d_training_data.pkl")
if not data_path.exists():
    print(f"Data file {data_path} not found. Please run src/generate_2d_data.py first.")
else:
    print(f"Loading data from {data_path}...")
    data = joblib.load(data_path)
    image_np = data["image"]
    contour_np = data["contour"]
    gt_contour_np = data["gt"]

In [ ]:
# Visualize Initial Data
plt.figure(figsize=(10, 10))
plt.imshow(image_np, cmap='gray')
plt.plot(contour_np[:, 1], contour_np[:, 0], 'r-', linewidth=2, label='Initial Contour')
if len(gt_contour_np) > 0:
    plt.plot(gt_contour_np[:, 1], gt_contour_np[:, 0], 'g--', linewidth=2, label='Ground Truth')
plt.legend()
plt.title("Initial Data")
plt.show()

In [ ]:
# 2. Smooth Contour
print("Smoothing initial contour...")
contour_smoothed = smooth_contour(contour_np, num_points=256)

plt.figure(figsize=(10, 10))
plt.imshow(image_np, cmap='gray')
plt.plot(contour_np[:, 1], contour_np[:, 0], 'r--', linewidth=1, label='Original')
plt.plot(contour_smoothed[:, 1], contour_smoothed[:, 0], 'b-', linewidth=2, label='Smoothed')
plt.legend()
plt.title("Smoothed Contour")
plt.show()

In [ ]:
# 3. Setup Optimization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Image: (1, 1, H, W)
image = torch.from_numpy(image_np).float().to(device).unsqueeze(0).unsqueeze(0)

# Contour: (N, 2) - Learnable
contour = torch.from_numpy(contour_smoothed).float().to(device)
contour.requires_grad = True

optimizer = torch.optim.Adam([contour], lr=0.5)

# Loss: 6px distance, 3px width (approx sigma=0.75)
criterion = BiGaussianLoss(peak_dist=6.0, sigma=0.75, profile_len=21).to(device)

In [ ]:
# 4. Optimization Loop
print("Starting optimization...")
history = []

for i in range(201):
    optimizer.zero_grad()

    normals = compute_normals(contour)
    profiles = sample_profiles(image, contour, normals, num_samples=21, width=3)

    data_loss = criterion(profiles)

    # Regularization
    v_next = torch.roll(contour, shifts=-1, dims=0)
    v_prev = torch.roll(contour, shifts=1, dims=0)

    # Laplacian smoothing
    laplacian = contour - 0.5 * (v_prev + v_next)
    lap_loss = (laplacian**2).sum(dim=-1).mean()

    # Edge length consistency
    edge_lengths = torch.norm(contour - v_next, dim=-1)
    edge_loss = ((edge_lengths - edge_lengths.mean()) ** 2).mean()

    loss = data_loss + 2.0 * lap_loss + 0.5 * edge_loss

    loss.backward()
    optimizer.step()

    history.append(loss.item())

    if i % 50 == 0:
        print(f"Iter {i:03d}: Loss={loss.item():.4f}")

        # Visualization
        current_contour = contour.detach().cpu().numpy()
        plt.figure(figsize=(8, 8))
        plt.imshow(image_np, cmap='gray')
        plt.plot(current_contour[:, 1], current_contour[:, 0], 'c-', linewidth=2)
        plt.title(f"Iteration {i}")
        plt.show()

In [ ]:
# Final Result
final_contour = contour.detach().cpu().numpy()

plt.figure(figsize=(12, 12))
plt.imshow(image_np, cmap='gray')
plt.plot(contour_np[:, 1], contour_np[:, 0], 'r--', linewidth=1.5, label='Initial')
plt.plot(final_contour[:, 1], final_contour[:, 0], 'b-', linewidth=2, label='Optimized')
if len(gt_contour_np) > 0:
    plt.plot(gt_contour_np[:, 1], gt_contour_np[:, 0], 'g:', linewidth=2, label='Ground Truth')

plt.legend()
plt.title("Final Optimization Result")
plt.show()